# NeuroProfile on Colab

Runs `test_encode.py` (OOM gate) and `batch_encode.py` (corpus grind) on a Colab GPU instead of the 8 GB 4060.

**What Colab changes vs. the Arch box (only three things really matter):**
1. **VRAM.** A T4 is 16 GB (free), an L4/A100 more (Pro) — double+ the 4060. The `predict()` OOM should just disappear. Confirm from the `[vram after predict chunk N]` line in the smoke test.
2. **Downloads.** Do **not** download YouTube on Colab — its datacenter IPs are hard-blocked (worse than your flagged-IP case). Pre-download clips on the 4060 (where the winning yt-dlp recipe works), drop the `.mp4`s in Drive, and feed **file paths**, not URLs. This sidesteps the entire yt-dlp/deno/EJS/PO-token/cookies saga on the encode side.
3. **Persistence.** `/content` is scratch and dies on disconnect. Qdrant + timelines go to Drive (constraint #10). The scripts already take these as CLI args, so no code edit.

The Arch-specific pain is *gone*: the ctranslate2 execstack ELF-patch is unnecessary (stock Ubuntu kernel allows exec-stack), and the flaky-network Llama download is fast here.

## 0 · GPU check

In [1]:
import torch
try:
    from tribev2 import TribeModel
    import neuralset
    print(">>> IMPORT OK — tribev2 loads on torch", torch.__version__)
except Exception as e:
    import traceback; traceback.print_exc()
    print("\n>>> IMPORT FAILED:", type(e).__name__, "-", e)


>>> IMPORT FAILED: ModuleNotFoundError - No module named 'tribev2'


Traceback (most recent call last):
  File "/tmp/ipykernel_5922/152939229.py", line 3, in <cell line: 0>
    from tribev2 import TribeModel
ModuleNotFoundError: No module named 'tribev2'


In [2]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

name, memory.total [MiB], memory.free [MiB]
NVIDIA L4, 23034 MiB, 22564 MiB


## 1 · Installs — run this, then **restart the runtime once**

All pip installs up front. After this cell: **Runtime ▸ Restart session**, then continue from Section 2. Restarting makes the pinned torch/numpy the versions that actually load (Colab preloads its own torch).

In [3]:
# numpy + torch must match tribev2's pins (numpy==2.2.6, torch>=2.5.1,<2.7).
# Installing the exact triple matches your handoff; if Colab's stock torch is already
# in [2.5.1, 2.7) you can skip the torch line and let tribev2 accept it.
!pip install -q numpy==2.2.6
!pip install -q torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121

# TRIBE v2 (not on PyPI). Its pins match ours, so it won't swap the torch triple.
!pip install -q "git+https://github.com/facebookresearch/tribev2.git"

# CPU-side pipeline deps (your repo's requirements, listed for a clean Colab env)
!pip install -q nibabel qdrant-client fastapi python-multipart uvicorn yt-dlp pytest scipy

# whisperX installed IN THIS env so TRIBE calls it directly (patched below) instead of
# uvx re-downloading ~3.5 GB every call. If this bumps torch, re-run the torch line above.
!pip install -q whisperx

print("installs done — now Runtime > Restart session, then run Section 2")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 122.4 MB/s eta 0:00:0000:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.2.6 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 2.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 99.6 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 95.7 MB/s eta 0:00:00:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 169.5 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 261.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 231.8 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

### ⚠️ Restart the runtime now (Runtime ▸ Restart session), then run Section 2 onward.

## 2 · Drive, repo, auth, weights

In [8]:
from google.colab import drive
drive.mount('/content/drive')

import os
NP = '/content/drive/MyDrive/neuroprofile'   # durable root (survives disconnects)
for sub in ('qdrant_data', 'data/timelines', 'clips'):
    os.makedirs(f'{NP}/{sub}', exist_ok=True)
print('durable root:', NP)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
durable root: /content/drive/MyDrive/neuroprofile


In [ ]:
# Get the repo onto FAST local scratch (/content), not Drive (Drive FUSE is slow for code).
# Make sure ica/ (frozen artifacts) and tests/reducer_reference.npz come along —
# reducer.py does np.load("ica/...") at import time and will crash without them.

# --- Option A: clone from GitHub (fill in your remote) ---
!git clone https://github.com/Mammbo/NeuroProfile.git /content/neuroprofile

# --- Option B: repo already in Drive — copy to scratch ---
# !cp -r /content/drive/MyDrive/neuroprofile/repo /content/neuroprofile

%cd /content/neuroprofile
!ls backend ica scripts

/bin/bash: line 1: you: No such file or directory
[Errno 2] No such file or directory: '/content/neuroprofile'
/content
ls: cannot access 'backend': No such file or directory
ls: cannot access 'ica': No such file or directory
ls: cannot access 'scripts': No such file or directory


In [ ]:
from huggingface_hub import login
login()   # HF token with access to gated meta-llama/Llama-3.2-3B

In [ ]:
# Colab's network is fast — this is the step that swung 47min->5.5hr on the 4060.
!hf download meta-llama/Llama-3.2-3B --include "*.safetensors" "config.json" "tokenizer*"
# older huggingface_hub: !huggingface-cli download meta-llama/Llama-3.2-3B --include "*.safetensors" "config.json" "tokenizer*"

In [ ]:
import os
os.environ["NLTK_ALLOW_PROXIED_URLOPEN"] = "1"
import nltk
for r in ["punkt_tab", "punkt"]:
    nltk.download(r)

# Patch TRIBE so get_events_dataframe calls whisperx directly instead of via `uvx`
# (uvx spins an isolated env and re-downloads ~3.5 GB every call -> looks frozen at 0%).
import tribev2, pathlib
et  = pathlib.Path(tribev2.__file__).parent / "eventstransforms.py"
src = et.read_text()
new = src.replace('["uvx", "whisperx"', '["whisperx"')
et.write_text(new)
print("uvx->whisperx patch:", "applied" if new != src else "NO CHANGE — check the pattern in eventstransforms.py")

# NOTE: no ctranslate2 execstack ELF-patch needed on Colab (Ubuntu kernel allows exec-stack).
# TRIBE's default whisperx is large-v3 fp16 on cuda. On a 16 GB T4 it may fit alongside TRIBE.
# If predict()+whisperx OOM even here, apply your 4060 edit (model="small", device="cpu",
# compute_type="int8") to eventstransforms.py.

## 3 · Feed clips (upload `.mp4`s to Drive, then encode)

Upload your pre-downloaded clips to `MyDrive/neuroprofile/clips/`. The **file route** needs no yt-dlp, no cookies, no download — `resolve_source` sniffs magic bytes and goes straight to `chunk_video`.

In [ ]:
%cd /content/neuroprofile
# Smoke test = your OOM gate. Watch [vram after predict chunk N] and the segment.start verdict.
# Use a >100 s clip to force 2 chunks so the segment.start check actually prints.
!python scripts/test_encode.py /content/drive/MyDrive/neuroprofile/clips/test.mp4

In [ ]:
# Build a corpus of FILE PATHS (not URLs) and grind it, persisting to Drive.
import glob, pathlib
clips = sorted(glob.glob('/content/drive/MyDrive/neuroprofile/clips/*.mp4'))
pathlib.Path('/content/corpus.txt').write_text("\n".join(clips))
print(len(clips), "clips -> /content/corpus.txt")

In [ ]:
!python scripts/batch_encode.py --corpus /content/corpus.txt \
    --qdrant-path   /content/drive/MyDrive/neuroprofile/qdrant_data \
    --timelines-dir /content/drive/MyDrive/neuroprofile/data/timelines

# Persisted to Drive => a Colab disconnect mid-corpus is fine: re-run this cell and
# already-encoded ids are skipped (db.get_video != None -> status "skip"). Resumable by design.
#
# Caveat: embedded Qdrant on the Drive FUSE mount can hit file-lock/IO quirks. If you see
# lock errors, point --qdrant-path at /content/qdrant_data (scratch) for speed, then
# `!cp -r /content/qdrant_data /content/drive/MyDrive/neuroprofile/` after the run.
# Timelines stream to Drive safely either way.